In [908]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import os
import requests
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from functools import reduce

In [909]:
uri = os.getenv('SBS_V1_MONGO_URI')

client = MongoClient(uri, server_api=ServerApi('1'))
db = client['SBSV1']

try:
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(e)

nba_games_historical_collection = db['nba_games_historical']
nba_team_aggregated_game_stats_historical_collection = db['nba_team_aggregated_game_stats_historical']
nba_game_player_stats_historical_collection = db['nba_game_player_stats_historical']
nba_player_aggregated_game_stats_historical_collection = db['nba_player_aggregated_game_stats_historical']
cached_web_api_response_collection = db['cached_web_api_response']

Pinged your deployment. You successfully connected to MongoDB!


In [910]:
############# SPORTS BETTING SANDBOX API ################

#########################################################
# get_event_odds ########################################
def get_event_odds(sports):
    url = 'https://sportsbettingsandboxapi.com/odds-api/events/get'
    response = requests.post(url, json={ 'sports': sports }).json()['data']
    return response
#########################################################

#########################################################
# get_event_odds ########################################
#[derive(Debug, Deserialize, Clone)]
#[serde(rename_all = 'camelCase')]
# pub struct GetOddsRequest {
#     pub sports: OddsApiSports,
#     pub regions: OddsApiRegions,
#     pub markets: Vec<String>,
#     pub odds_format: OddsFormat,
#     pub bookmakers: Vec<Bookmakers>
# }
def get_odds(req):
    url = 'https://sportsbettingsandboxapi.com/odds-api/odds/get'
    response = requests.post(url, json=req).json()['data']['events']
    return response
#########################################################

In [911]:
################### MONGO FUNCS #########################

#########################################################
# get_historical_nba_game_objs_from_season ##############
def get_historical_nba_game_objs_from_season(season):
    return list(nba_games_historical_collection.find({ 'season': season }))
#########################################################

#########################################################
# get_historical_nba_player_aggregated_game_stats_from_season
def get_historical_nba_player_aggregated_game_stats_from_season(season, season_type):
    return list(nba_player_aggregated_game_stats_historical_collection.find({ 'season': season, 'seasonType': season_type }))
#########################################################

In [912]:
################### HELPER FUNCS ########################

#########################################################
# transform_player_game_stats_to_df #####################
def transform_player_game_stats_objs_to_df(player_game_stats_objs):
    stats = [stat for player in player_game_stats_objs for stat in player['playerStats'].values()]
    return pd.DataFrame(stats)
#########################################################

In [913]:
##################### ML FUNCS ##########################

#########################################################
# enrich_player_stats_df_with_player_role ###############
def enrich_player_stats_df_with_player_role(player_stats_df):
    player_stats_for_clustering = ['points', 'assists', 'totReb', 'fgm', 'fga', 'tpm', 'tpa', 'ftm', 'fta', 'turnovers', 'blocks', 'steals']
    
    player_stats_df['date'] = player_stats_df['dateStart'].str.split('T').str[0]
    unique_dates = sorted(player_stats_df['date'].unique())

    role_cluster_rows = []
    for date in unique_dates:
        # Filter out games with zero minutes and keep games before date
        df = player_stats_df[(player_stats_df['date'] < date) & (player_stats_df['min'] > 0)].copy()
       
        if df.empty:
            continue            
            
        # Normalize stats per 36 minutes
        for stat in player_stats_for_clustering:
            df[stat] = df[stat] * (36 / df['min'])
    
        # Group by playerId and calculate mean
        all_players_avg_stats = (
            df
            .sort_values('dateStart', ascending=False)
            .groupby('playerId')[player_stats_for_clustering]
            .mean()
            .reset_index()
        )
    
        all_players_avg_stats = all_players_avg_stats.dropna()
    
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(all_players_avg_stats.drop('playerId', axis=1))
    
        kmeans = KMeans(n_clusters=5, random_state=42)
        all_players_avg_stats['roleCluster'] = kmeans.fit_predict(X_scaled)
        all_players_avg_stats['date'] = date
        
        role_cluster_rows.append(all_players_avg_stats[['playerId', 'roleCluster', 'date']])

    # Combine all per-date role cluster snapshots
    full_role_cluster_df = pd.concat(role_cluster_rows, ignore_index=True)

    # Ensure numeric consistency in merge keys
    player_stats_df['playerId'] = player_stats_df['playerId'].astype('int64')
    full_role_cluster_df['playerId'] = full_role_cluster_df['playerId'].astype('int64')

    full_role_cluster_df['date'] = pd.to_datetime(full_role_cluster_df['date']).dt.tz_localize(None)
    full_role_cluster_df['date'] = pd.to_datetime(full_role_cluster_df['date'], unit='ms')
    
    player_stats_df['date'] = pd.to_datetime(player_stats_df['date']).dt.tz_localize(None)
    player_stats_df['date'] = pd.to_datetime(player_stats_df['date'], unit='ms')

    enriched_df = pd.merge_asof(
        player_stats_df.sort_values('date'),
        full_role_cluster_df.sort_values('date'),
        on='date',
        by=['playerId'],
        direction='backward'  # use the most recent past date
    )
    
    return enriched_df[enriched_df['roleCluster'].notna()]
#########################################################


#########################################################
# enrich_player_stats_df_with_dvp_stats #################
def enrich_player_stats_df_with_dvp_stats(player_stats_df):
    stats = ['points', 'assists', 'totReb', 'fgm', 'fga', 'tpm', 'tpa', 'turnovers', 'blocks', 'steals']
    rolling_windows = [10]

    # Extract date and ensure proper sorting
    player_stats_df['date'] = player_stats_df['dateStart'].str.split('T').str[0]
    player_stats_df = player_stats_df.sort_values(['opponentTeamId', 'roleCluster', 'date'])

    dvp_rows = []
    unique_dates = sorted(player_stats_df['date'].unique())

    for date in unique_dates:
        # Filter past games for each date
        df = player_stats_df[player_stats_df['date'] < date]
        if df.empty:
            continue

        rolling_dvp_for_date_rows = []

        # ROLLING AVG for each window
        for window in rolling_windows:
            dvp_team_cluster = (
                df.groupby(['opponentTeamId', 'roleCluster'])[stats]
                .apply(lambda x: x.rolling(window=window, min_periods=1).mean().iloc[-1])
                .reset_index()
            )
            dvp_team_cluster = dvp_team_cluster.rename(columns={stat: f'{window}_g_{stat}_team_dvp' for stat in stats})

            dvp_league_cluster = (
                df.groupby('roleCluster')[stats]
                .apply(lambda x: x.rolling(window=window, min_periods=1).mean().iloc[-1])
                .reset_index()
            )
            dvp_league_cluster = dvp_league_cluster.rename(columns={stat: f'{window}_g_{stat}_league_dvp' for stat in stats})

            # Merge to calculate pct_diff
            dvp_full = dvp_team_cluster.merge(dvp_league_cluster, on='roleCluster')

            for stat in stats:
                dvp_full[f'{window}_g_{stat}_dvp_pct_diff'] = (
                    (dvp_full[f'{window}_g_{stat}_team_dvp'] - dvp_full[f'{window}_g_{stat}_league_dvp']) /
                    dvp_full[f'{window}_g_{stat}_league_dvp']
                )

            rolling_dvp_for_date_rows.append(dvp_full)

        # EXPANDING AVG
        dvp_team_cluster = df.groupby(['opponentTeamId', 'roleCluster'])[stats].mean().reset_index()
        dvp_team_cluster = dvp_team_cluster.rename(columns={stat: f'all_g_{stat}_team_dvp' for stat in stats})

        dvp_league_cluster = df.groupby('roleCluster')[stats].mean().reset_index()
        dvp_league_cluster = dvp_league_cluster.rename(columns={stat: f'all_g_{stat}_league_dvp' for stat in stats})

        dvp_full = dvp_team_cluster.merge(dvp_league_cluster, on='roleCluster')

        for stat in stats:
            dvp_full[f'all_g_{stat}_dvp_pct_diff'] = (
                (dvp_full[f'all_g_{stat}_team_dvp'] - dvp_full[f'all_g_{stat}_league_dvp']) /
                dvp_full[f'all_g_{stat}_league_dvp']
            )

        rolling_dvp_for_date_rows.append(dvp_full)

        # Merge all for the date
        rolling_dvp_for_date_df = reduce(
            lambda left, right: pd.merge(left, right, on=['opponentTeamId', 'roleCluster']),
            rolling_dvp_for_date_rows
        )
        rolling_dvp_for_date_df['date'] = date
        dvp_rows.append(rolling_dvp_for_date_df)

    # Combine all per-date DvP snapshots
    full_dvp_df = pd.concat(dvp_rows, ignore_index=True)

    # Drop rows where keys are missing
    player_stats_df = player_stats_df.dropna(subset=['opponentTeamId', 'roleCluster', 'date'])
    full_dvp_df = full_dvp_df.dropna(subset=['opponentTeamId', 'roleCluster', 'date'])


    player_stats_df['date'] = pd.to_datetime(player_stats_df['date']).dt.tz_localize(None)
    player_stats_df['date'] = pd.to_datetime(player_stats_df['date'], unit='ms')
    
    full_dvp_df['date'] = pd.to_datetime(full_dvp_df['date']).dt.tz_localize(None)
    full_dvp_df['date'] = pd.to_datetime(full_dvp_df['date'], unit='ms')

    # Ensure numeric consistency in merge keys
    player_stats_df['opponentTeamId'] = player_stats_df['opponentTeamId'].astype('int64')
    player_stats_df['roleCluster'] = player_stats_df['roleCluster'].astype('int64')
    full_dvp_df['opponentTeamId'] = full_dvp_df['opponentTeamId'].astype('int64')
    full_dvp_df['roleCluster'] = full_dvp_df['roleCluster'].astype('int64')

    # Merge back
    enriched_df = pd.merge_asof(
        player_stats_df.sort_values('date'),
        full_dvp_df.sort_values('date'),
        on='date',
        by=['opponentTeamId', 'roleCluster'],
        direction='backward'
    )

    return enriched_df
#########################################################

#########################################################
# enrich_player_stats_df_with_rolling_stats #############
def enrich_player_stats_df_with_rolling_stats(player_stats_df):
    # Raw stats to be processed
    stats = ['points', 'assists', 'totReb', 'fgm', 'fga', 'tpm', 'tpa', 'turnovers', 'blocks', 'steals']
    rolling_windows = [5, 10, 20]

    # Extract date and ensure proper sorting
    player_stats_df['date'] = player_stats_df['dateStart'].str.split('T').str[0]
    player_stats_df = player_stats_df.sort_values(['playerId', 'date'])

    rolling_stats_rows = []
    unique_dates = sorted(player_stats_df['date'].unique())

    for date in unique_dates:
        # Filter past games for each date
        df = player_stats_df[player_stats_df['date'] < date]
        if df.empty:
            continue

        rolling_stats_for_date_rows = []

        # Calculate rolling averages and stds
        for window in rolling_windows:
            rolling_avg_df = (
                df.groupby('playerId')[stats]
                .apply(lambda x: x.rolling(window=window, min_periods=1).mean().iloc[-1])
                .reset_index()
            )
            rolling_avg_df = rolling_avg_df.rename(columns={stat: f'{window}_g_{stat}_roll_avg' for stat in stats})

            rolling_std_df = (
                df.groupby('playerId')[stats]
                .apply(lambda x: x.rolling(window=window, min_periods=2).std().iloc[-1])
                .reset_index()
            )
            rolling_std_df = rolling_std_df.rename(columns={stat: f'{window}_g_{stat}_roll_std' for stat in stats})

            rolling_stats_for_date_rows.extend([rolling_avg_df, rolling_std_df])

        # Expanding average (all games up to the current date)
        expanding_avg_df = (
            df.groupby('playerId')[stats]
            .mean()
            .reset_index()
        ).rename(columns={stat: f'all_g_{stat}_roll_avg' for stat in stats})

        rolling_stats_for_date_rows.append(expanding_avg_df)

        # Merge all rolling stats for the current date
        rolling_stats_for_date_df = reduce(
            lambda left, right: pd.merge(left, right, on='playerId'), 
            rolling_stats_for_date_rows
        )
        rolling_stats_for_date_df['date'] = date
        rolling_stats_rows.append(rolling_stats_for_date_df)

    # Combine all per-date rolling stats
    full_rolling_stats_df = pd.concat(rolling_stats_rows, ignore_index=True)

    # Z-Score Calculation relative to the 20-game rolling average
    for window in rolling_windows[:len(rolling_windows)-1]:  # Only 5 and 10 game windows use 20-game avg for Z-score
        for stat in stats:
            z_score_col = f'{window}_g_{stat}_z_score'
            avg_col = f'{window}_g_{stat}_roll_avg'
            std_col = f'{window}_g_{stat}_roll_std'
            reference_avg_col = f'{rolling_windows[len(rolling_windows)-1]}_g_{stat}_roll_avg'

            # Ensure the 20-game rolling avg exists for each row
            if reference_avg_col in full_rolling_stats_df.columns:
                full_rolling_stats_df[z_score_col] = (
                    (full_rolling_stats_df[avg_col] - full_rolling_stats_df[reference_avg_col]) /
                    full_rolling_stats_df[std_col]
                )
                # Handle division by zero or NaN values
                full_rolling_stats_df[z_score_col] = full_rolling_stats_df[z_score_col].fillna(0)


    player_stats_df['date'] = pd.to_datetime(player_stats_df['date']).dt.tz_localize(None)
    player_stats_df['date'] = pd.to_datetime(player_stats_df['date'], unit='ms')
    
    full_rolling_stats_df['date'] = pd.to_datetime(full_rolling_stats_df['date']).dt.tz_localize(None)
    full_rolling_stats_df['date'] = pd.to_datetime(full_rolling_stats_df['date'], unit='ms')
    
    player_stats_df['playerId'] = player_stats_df['playerId'].astype('int64')
    full_rolling_stats_df['playerId'] = full_rolling_stats_df['playerId'].astype('int64')

    # Merge back with the original DataFrame
    enriched_df = pd.merge_asof(
        player_stats_df.sort_values('date'),
        full_rolling_stats_df.sort_values('date'),
        on='date',
        by='playerId',
        direction='backward'
    )


    return enriched_df
#########################################################

In [914]:
################# ML PIPELINE FUNCS #####################
def get_training_data_for_sbs_nba_ensemble_model_1(season, season_type):
    # for each step, the aggregated data set should be cached
    print(f'getting player data for season: {season}, season_type: {season_type}')
    player_stats_objs = get_historical_nba_player_aggregated_game_stats_from_season(season, season_type)
    player_stats_df = transform_player_game_stats_objs_to_df(player_stats_objs)
    print('Enriching feature map with role clustering')
    enriched_player_stats_df = enrich_player_stats_df_with_player_role(player_stats_df)
    print('Enriching feature map with DVP stats')
    enriched_player_stats_df = enrich_player_stats_df_with_dvp_stats(enriched_player_stats_df)
    print('Enriching feature map with rolling stats, std, and z-score stats')
    enriched_player_stats_df = enrich_player_stats_df_with_rolling_stats(enriched_player_stats_df)
    print()
    enriched_player_stats_df.fillna(0, inplace=True)
    # print(enriched_player_stats_df.head(1).to_json(orient='records', indent=2))
    return enriched_player_stats_df
#########################################################

In [915]:
nba_player_training_data_2024 = get_training_data_for_sbs_nba_ensemble_model_1(2024, 'ALL')
nba_player_training_data_2023 = get_training_data_for_sbs_nba_ensemble_model_1(2023, 'ALL')
nba_player_training_data_2022 = get_training_data_for_sbs_nba_ensemble_model_1(2022, 'ALL')

all_nba_player_training_data = pd.concat([nba_player_training_data_2024, nba_player_training_data_2023, nba_player_training_data_2022], ignore_index=True)

print(all_nba_player_training_data.shape)

getting player data for season: 2024, season_type: ALL
Enriching feature map with role clustering
Enriching feature map with DVP stats
Enriching feature map with rolling stats, std, and z-score stats

getting player data for season: 2023, season_type: ALL
Enriching feature map with role clustering
Enriching feature map with DVP stats
Enriching feature map with rolling stats, std, and z-score stats

getting player data for season: 2022, season_type: ALL
Enriching feature map with role clustering
Enriching feature map with DVP stats
Enriching feature map with rolling stats, std, and z-score stats

(77468, 179)
